# Fine-Tuning for Bengali Coreference Resolution
## Frozen Encoder + Pairwise Classifier

**Author:** Zenith  
**Date:** December 2025  
**Purpose:** Fine-tune 5 models using frozen encoder + trainable pairwise classifier

---

## Methodology

**Architecture:**
- Encoder: Frozen (all transformer layers)
- Trainable: Pairwise MLP classifier
- Loss: Binary Cross-Entropy

**Training Protocol:**
- Negative pairs subsampled at 1:4 ratio per epoch
- Early stopping based on dev CoNLL F1 (patience=3)

**Evaluation Protocol:**
- Threshold tuned on dev set: [0.1, 0.9], step 0.05
- Same Graph-CC clustering as zero-shot baseline
- Final evaluation on test set with fixed threshold

---

## Data

| Split | File | Documents |
|-------|------|----------|
| Train | transmucores_bencoref_train.conll | 141 |
| Dev | dev.conll | 10 |
| Test | test.conll | 71 |

In [7]:
"""
Cell 1: Setup and Imports
"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")

# Core imports
import json
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
from itertools import combinations
from datetime import datetime
from tqdm import tqdm
import time
import gc
import copy

# ML imports
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import networkx as nx

# Set random seeds
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Environment check
print("=" * 80)
print("FINE-TUNING: FROZEN ENCODER + PAIRWISE CLASSIFIER")
print("=" * 80)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nEnvironment:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 80)

FINE-TUNING: FROZEN ENCODER + PAIRWISE CLASSIFIER
Timestamp: 2025-12-17 02:39:35

Environment:
  PyTorch: 2.9.0+cu128
  CUDA available: True
  GPU: NVIDIA L4
  GPU Memory: 23.58 GB


In [8]:
"""
Cell 2: Configuration
"""

# =============================================================================
# PATHS
# =============================================================================
BASE_DIR = Path('/teamspace/studios/this_studio/final_experiments')
TRAIN_FILE = BASE_DIR / 'transmucores_bencoref_train.conll'
DEV_FILE = BASE_DIR / 'dev.conll'
TEST_FILE = BASE_DIR / 'test.conll'
OUTPUT_DIR = BASE_DIR / 'experiment_1_finetuned'
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# =============================================================================
# MODEL CONFIGURATIONS
# =============================================================================
@dataclass
class ModelConfig:
    name: str
    hf_model_id: str
    category: str
    hidden_dim: int

MODELS = [
    ModelConfig(
        name="mBERT",
        hf_model_id="google-bert/bert-base-multilingual-cased",
        category="multilingual",
        hidden_dim=768
    ),
    ModelConfig(
        name="BanglaBERT-Base",
        hf_model_id="csebuetnlp/banglabert",
        category="bengali-specific",
        hidden_dim=768
    ),
    ModelConfig(
        name="RemBERT",
        hf_model_id="google/rembert",
        category="low-resource",
        hidden_dim=1152
    ),
    ModelConfig(
        name="MuRIL-Large",
        hf_model_id="google/muril-large-cased",
        category="indic-focused",
        hidden_dim=1024
    ),
    ModelConfig(
        name="BERT-Base-Uncased",
        hf_model_id="google-bert/bert-base-uncased",
        category="control",
        hidden_dim=768
    ),
]

# =============================================================================
# TRAINING HYPERPARAMETERS
# =============================================================================
@dataclass
class TrainingConfig:
    # Negative sampling
    neg_ratio: int = 4  # 1:4 positive to negative
    
    # Training
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 10
    batch_size: int = 64
    
    # Early stopping
    patience: int = 3
    
    # Threshold tuning
    threshold_min: float = 0.1
    threshold_max: float = 0.9
    threshold_step: float = 0.05
    
    # MLP architecture
    mlp_hidden: int = 512
    mlp_dropout: float = 0.3

CONFIG = TrainingConfig()

# Print configuration
print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"\nData:")
print(f"  Train: {TRAIN_FILE}")
print(f"  Dev:   {DEV_FILE}")
print(f"  Test:  {TEST_FILE}")
print(f"\nTraining:")
print(f"  Negative sampling ratio: 1:{CONFIG.neg_ratio}")
print(f"  Learning rate: {CONFIG.learning_rate}")
print(f"  Batch size: {CONFIG.batch_size}")
print(f"  Epochs: {CONFIG.epochs}")
print(f"  Early stopping patience: {CONFIG.patience}")
print(f"\nThreshold tuning:")
print(f"  Range: [{CONFIG.threshold_min}, {CONFIG.threshold_max}]")
print(f"  Step: {CONFIG.threshold_step}")
print(f"\nModels ({len(MODELS)}):")
for m in MODELS:
    print(f"  • {m.name} ({m.category})")
print("=" * 80)

CONFIGURATION

Data:
  Train: /teamspace/studios/this_studio/final_experiments/transmucores_bencoref_train.conll
  Dev:   /teamspace/studios/this_studio/final_experiments/dev.conll
  Test:  /teamspace/studios/this_studio/final_experiments/test.conll

Training:
  Negative sampling ratio: 1:4
  Learning rate: 0.001
  Batch size: 64
  Epochs: 10
  Early stopping patience: 3

Threshold tuning:
  Range: [0.1, 0.9]
  Step: 0.05

Models (5):
  • mBERT (multilingual)
  • BanglaBERT-Base (bengali-specific)
  • RemBERT (low-resource)
  • MuRIL-Large (indic-focused)
  • BERT-Base-Uncased (control)


In [9]:
"""
Cell 3: CoNLL Parser
"""

def parse_conll_file(filepath: Path) -> List[Dict]:
    """
    Parse CoNLL-2012 format file and extract documents with mentions and clusters.
    """
    documents = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split by document
    doc_pattern = r'#begin document \((.+?)\).*?\n(.*?)#end document'
    doc_matches = re.findall(doc_pattern, content, re.DOTALL)
    
    for doc_id, doc_content in doc_matches:
        sentences = []
        current_sentence = []
        
        cluster_mentions = defaultdict(list)
        open_mentions = defaultdict(list)
        
        word_idx_in_sent = 0
        sent_idx = 0
        
        for line in doc_content.strip().split('\n'):
            if not line.strip():
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                    sent_idx += 1
                    word_idx_in_sent = 0
                continue
            
            parts = line.split()
            if len(parts) < 4:
                continue
            
            token = parts[3]
            coref_col = parts[-1]
            
            current_sentence.append(token)
            
            if coref_col != '-' and coref_col != '_':
                annotations = coref_col.split('|')
                for ann in annotations:
                    single_match = re.match(r'^\((\d+)\)$', ann)
                    if single_match:
                        cluster_id = int(single_match.group(1))
                        cluster_mentions[cluster_id].append(
                            (sent_idx, word_idx_in_sent, word_idx_in_sent)
                        )
                        continue
                    
                    open_match = re.match(r'^\((\d+)$', ann)
                    if open_match:
                        cluster_id = int(open_match.group(1))
                        open_mentions[cluster_id].append((sent_idx, word_idx_in_sent))
                        continue
                    
                    close_match = re.match(r'^(\d+)\)$', ann)
                    if close_match:
                        cluster_id = int(close_match.group(1))
                        if open_mentions[cluster_id]:
                            start_sent, start_word = open_mentions[cluster_id].pop()
                            if start_sent == sent_idx:
                                cluster_mentions[cluster_id].append(
                                    (sent_idx, start_word, word_idx_in_sent)
                                )
            
            word_idx_in_sent += 1
        
        if current_sentence:
            sentences.append(current_sentence)
        
        # Convert to mention list and cluster indices
        all_mentions = []
        mention_to_idx = {}
        clusters = []
        
        for cluster_id, mention_spans in cluster_mentions.items():
            cluster_indices = []
            for sent_idx, start, end in mention_spans:
                key = (sent_idx, start, end)
                if key not in mention_to_idx:
                    mention_to_idx[key] = len(all_mentions)
                    all_mentions.append({
                        'sentence_idx': sent_idx,
                        'start': start,
                        'end': end
                    })
                cluster_indices.append(mention_to_idx[key])
            if cluster_indices:
                clusters.append(cluster_indices)
        
        if all_mentions:  # Only add docs with mentions
            documents.append({
                'id': doc_id,
                'sentences': sentences,
                'mentions': all_mentions,
                'clusters': clusters
            })
    
    return documents

# Load all data
print("Loading data...")
train_docs = parse_conll_file(TRAIN_FILE)
dev_docs = parse_conll_file(DEV_FILE)
test_docs = parse_conll_file(TEST_FILE)

print(f"\n" + "=" * 80)
print("DATA LOADED")
print("=" * 80)
print(f"  Train: {len(train_docs)} docs, {sum(len(d['mentions']) for d in train_docs):,} mentions")
print(f"  Dev:   {len(dev_docs)} docs, {sum(len(d['mentions']) for d in dev_docs):,} mentions")
print(f"  Test:  {len(test_docs)} docs, {sum(len(d['mentions']) for d in test_docs):,} mentions")
print("=" * 80)

Loading data...

DATA LOADED
  Train: 141 docs, 27,135 mentions
  Dev:   10 docs, 447 mentions
  Test:  71 docs, 3,308 mentions


In [10]:
"""
Cell 4: CoNLL Evaluation Metrics
"""

class CorefMetrics:
    """Standard CoNLL-2012 coreference metrics."""
    
    @staticmethod
    def muc(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Tuple[float, float, float]:
        def links(clusters, reference_clusters):
            total_mentions = sum(len(c) for c in clusters)
            total_partitions = 0
            for cluster in clusters:
                cluster_set = set(cluster)
                partitions = sum(1 for ref in reference_clusters if cluster_set & set(ref))
                total_partitions += partitions
            return total_mentions - total_partitions
        
        gold_mentions = sum(len(c) for c in gold_clusters)
        gold_links = gold_mentions - len(gold_clusters)
        recall = links(gold_clusters, pred_clusters) / gold_links if gold_links > 0 else 0.0
        
        pred_mentions = sum(len(c) for c in pred_clusters)
        pred_links = pred_mentions - len(pred_clusters)
        precision = links(pred_clusters, gold_clusters) / pred_links if pred_links > 0 else 0.0
        
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return precision, recall, f1
    
    @staticmethod
    def b_cubed(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Tuple[float, float, float]:
        mention_to_gold = {}
        for cluster in gold_clusters:
            cluster_set = frozenset(cluster)
            for mention in cluster:
                mention_to_gold[mention] = cluster_set
        
        mention_to_pred = {}
        for cluster in pred_clusters:
            cluster_set = frozenset(cluster)
            for mention in cluster:
                mention_to_pred[mention] = cluster_set
        
        all_mentions = set(mention_to_gold.keys()) | set(mention_to_pred.keys())
        if not all_mentions:
            return 0.0, 0.0, 0.0
        
        total_precision = 0.0
        total_recall = 0.0
        
        for mention in all_mentions:
            gold_cluster = mention_to_gold.get(mention, frozenset([mention]))
            pred_cluster = mention_to_pred.get(mention, frozenset([mention]))
            intersection = len(gold_cluster & pred_cluster)
            
            if len(pred_cluster) > 0:
                total_precision += intersection / len(pred_cluster)
            if len(gold_cluster) > 0:
                total_recall += intersection / len(gold_cluster)
        
        precision = total_precision / len(all_mentions)
        recall = total_recall / len(all_mentions)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return precision, recall, f1
    
    @staticmethod
    def ceaf_e(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Tuple[float, float, float]:
        if not gold_clusters or not pred_clusters:
            return 0.0, 0.0, 0.0
        
        n_gold, n_pred = len(gold_clusters), len(pred_clusters)
        similarity_matrix = np.zeros((n_gold, n_pred))
        
        for i, gold_cluster in enumerate(gold_clusters):
            gold_set = set(gold_cluster)
            for j, pred_cluster in enumerate(pred_clusters):
                pred_set = set(pred_cluster)
                intersection = len(gold_set & pred_set)
                if len(gold_set) + len(pred_set) > 0:
                    similarity_matrix[i, j] = 2 * intersection / (len(gold_set) + len(pred_set))
        
        row_ind, col_ind = linear_sum_assignment(-similarity_matrix)
        total_similarity = similarity_matrix[row_ind, col_ind].sum()
        
        recall = total_similarity / n_gold if n_gold > 0 else 0.0
        precision = total_similarity / n_pred if n_pred > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return precision, recall, f1
    
    @staticmethod
    def evaluate(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Dict[str, float]:
        gold_non_singleton = [c for c in gold_clusters if len(c) > 1]
        pred_non_singleton = [c for c in pred_clusters if len(c) > 1]
        
        muc_p, muc_r, muc_f1 = CorefMetrics.muc(gold_non_singleton, pred_non_singleton)
        b3_p, b3_r, b3_f1 = CorefMetrics.b_cubed(gold_clusters, pred_clusters)
        ceaf_p, ceaf_r, ceaf_f1 = CorefMetrics.ceaf_e(gold_clusters, pred_clusters)
        
        conll_f1 = (muc_f1 + b3_f1 + ceaf_f1) / 3
        
        return {
            'MUC_F1': muc_f1 * 100,
            'B3_F1': b3_f1 * 100,
            'CEAF_F1': ceaf_f1 * 100,
            'CoNLL_F1': conll_f1 * 100
        }

print("✓ CorefMetrics defined")

✓ CorefMetrics defined


In [11]:
"""
Cell 5: Graph-CC Clustering (same as zero-shot)
"""

def graph_connected_components(similarity_matrix: np.ndarray, threshold: float) -> List[List[int]]:
    """
    Graph-based Connected Components clustering.
    Same algorithm as zero-shot baseline for fair comparison.
    """
    n_mentions = similarity_matrix.shape[0]
    if n_mentions == 0:
        return []
    if n_mentions == 1:
        return [[0]]
    
    G = nx.Graph()
    G.add_nodes_from(range(n_mentions))
    
    for i in range(n_mentions):
        for j in range(i + 1, n_mentions):
            if similarity_matrix[i, j] >= threshold:
                G.add_edge(i, j)
    
    clusters = [list(component) for component in nx.connected_components(G)]
    return clusters

print("✓ Graph-CC clustering defined")

✓ Graph-CC clustering defined


In [12]:
"""
Cell 6: Pairwise Classifier Model
"""

class PairwiseCorefScorer(nn.Module):
    """
    Frozen Encoder + Trainable Pairwise Classifier.
    
    Architecture:
    - Encoder: Frozen (all parameters)
    - Pairwise MLP: Trainable
    
    Pair features: [emb_i || emb_j || emb_i * emb_j || |emb_i - emb_j|]
    """
    
    def __init__(self, encoder, tokenizer, hidden_dim: int, 
                 mlp_hidden: int = 512, dropout: float = 0.3, device='cuda'):
        super().__init__()
        self.encoder = encoder
        self.tokenizer = tokenizer
        self.device = device
        self.hidden_dim = hidden_dim
        
        # Freeze encoder
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.encoder.eval()
        
        # Span embedding dimension: [start || end || mean]
        span_dim = hidden_dim * 3
        
        # Pair feature dimension: [emb_i || emb_j || emb_i*emb_j || |emb_i-emb_j|]
        pair_dim = span_dim * 4
        
        # Pairwise classifier MLP
        self.pairwise_mlp = nn.Sequential(
            nn.Linear(pair_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden // 2, 1),
            nn.Sigmoid()
        )
        
        # Move MLP to device
        self.pairwise_mlp = self.pairwise_mlp.to(device)
    
    def get_span_embedding(self, sentence: List[str], start: int, end: int) -> torch.Tensor:
        """
        Extract span embedding [start || end || mean] from frozen encoder.
        """
        # Tokenize
        encoding = self.tokenizer(
            sentence,
            is_split_into_words=True,
            return_tensors='pt',
            truncation=True,
            max_length=512,
            padding=True
        )
        encoding = {k: v.to(self.device) for k, v in encoding.items()}
        
        # Get word_ids mapping
        word_ids = self.tokenizer(
            sentence,
            is_split_into_words=True,
            truncation=True,
            max_length=512
        ).word_ids()
        
        # Find token positions
        positions = []
        for tok_idx, word_idx in enumerate(word_ids):
            if word_idx is not None and start <= word_idx <= end:
                positions.append(tok_idx)
        
        if not positions:
            positions = [1]  # Fallback: skip [CLS]
        
        # Get hidden states (no grad - encoder is frozen)
        with torch.no_grad():
            outputs = self.encoder(**encoding)
            hidden_states = outputs.last_hidden_state[0]  # (seq_len, hidden_dim)
        
        # Span embedding: [start || end || mean]
        start_emb = hidden_states[positions[0]]
        end_emb = hidden_states[positions[-1]]
        mean_emb = hidden_states[positions].mean(dim=0)
        
        return torch.cat([start_emb, end_emb, mean_emb])
    
    def forward(self, emb_i: torch.Tensor, emb_j: torch.Tensor) -> torch.Tensor:
        """
        Score a pair of mention embeddings.
        Returns probability of being coreferent.
        """
        # Pair features
        pair_features = torch.cat([
            emb_i,
            emb_j,
            emb_i * emb_j,              # Element-wise product
            torch.abs(emb_i - emb_j)    # Absolute difference
        ], dim=-1)
        
        return self.pairwise_mlp(pair_features)
    
    def get_all_mention_embeddings(self, doc: Dict) -> torch.Tensor:
        """
        Extract embeddings for all mentions in a document.
        Returns tensor of shape (n_mentions, span_dim).
        """
        embeddings = []
        for mention in doc['mentions']:
            sent = doc['sentences'][mention['sentence_idx']]
            emb = self.get_span_embedding(sent, mention['start'], mention['end'])
            embeddings.append(emb)
        return torch.stack(embeddings)
    
    def compute_pairwise_scores(self, mention_embeddings: torch.Tensor) -> np.ndarray:
        """
        Compute pairwise scores for all mention pairs.
        Returns similarity matrix for clustering.
        """
        n_mentions = mention_embeddings.shape[0]
        scores = np.zeros((n_mentions, n_mentions))
        
        # Set diagonal to 1 (self-similarity)
        np.fill_diagonal(scores, 1.0)
        
        # Compute pairwise scores
        for i in range(n_mentions):
            for j in range(i + 1, n_mentions):
                emb_i = mention_embeddings[i].unsqueeze(0)
                emb_j = mention_embeddings[j].unsqueeze(0)
                
                with torch.no_grad():
                    score = self.forward(emb_i, emb_j).item()
                
                scores[i, j] = score
                scores[j, i] = score
        
        return scores
    
    def trainable_parameters(self):
        """Return only trainable parameters (MLP)."""
        return self.pairwise_mlp.parameters()
    
    def save_mlp(self, path: Path):
        """Save only the MLP weights."""
        torch.save(self.pairwise_mlp.state_dict(), path)
    
    def load_mlp(self, path: Path):
        """Load MLP weights."""
        self.pairwise_mlp.load_state_dict(torch.load(path))

print("✓ PairwiseCorefScorer defined")

# Count trainable params for a sample model
sample_span_dim = 768 * 3
sample_pair_dim = sample_span_dim * 4
mlp_params = sample_pair_dim * 512 + 512 + 512 * 256 + 256 + 256 * 1 + 1
print(f"  Estimated trainable parameters (base model): ~{mlp_params:,}")

✓ PairwiseCorefScorer defined
  Estimated trainable parameters (base model): ~4,850,689


In [13]:
"""
Cell 7: Training Data Sampling
"""

def create_mention_to_cluster(clusters: List[List[int]]) -> Dict[int, int]:
    """Map mention index to cluster index."""
    mention_to_cluster = {}
    for cluster_idx, cluster in enumerate(clusters):
        for mention_idx in cluster:
            mention_to_cluster[mention_idx] = cluster_idx
    return mention_to_cluster


def sample_training_pairs(doc: Dict, neg_ratio: int = 4) -> List[Tuple[int, int, int]]:
    """
    Sample training pairs from a document.
    
    Returns list of (mention_i, mention_j, label) tuples.
    Label: 1 = coreferent, 0 = not coreferent
    
    Negative pairs are subsampled at 1:neg_ratio.
    """
    n_mentions = len(doc['mentions'])
    if n_mentions < 2:
        return []
    
    mention_to_cluster = create_mention_to_cluster(doc['clusters'])
    
    # Positive pairs: same cluster
    pos_pairs = []
    for cluster in doc['clusters']:
        if len(cluster) >= 2:
            for i, j in combinations(cluster, 2):
                pos_pairs.append((i, j, 1))
    
    # All possible pairs
    all_pairs = set(combinations(range(n_mentions), 2))
    pos_set = {(i, j) for i, j, _ in pos_pairs}
    neg_candidates = list(all_pairs - pos_set)
    
    # Subsample negatives
    n_neg = min(len(pos_pairs) * neg_ratio, len(neg_candidates))
    if n_neg > 0:
        neg_sampled = random.sample(neg_candidates, n_neg)
        neg_pairs = [(i, j, 0) for i, j in neg_sampled]
    else:
        neg_pairs = []
    
    return pos_pairs + neg_pairs


def create_epoch_pairs(docs: List[Dict], neg_ratio: int = 4) -> List[Tuple[Dict, int, int, int]]:
    """
    Create training pairs for one epoch.
    Re-samples negatives each epoch for variety.
    
    Returns list of (doc, mention_i, mention_j, label) tuples.
    """
    all_pairs = []
    for doc in docs:
        pairs = sample_training_pairs(doc, neg_ratio)
        for i, j, label in pairs:
            all_pairs.append((doc, i, j, label))
    
    random.shuffle(all_pairs)
    return all_pairs


# Test sampling
print("Testing pair sampling...")
sample_pairs = sample_training_pairs(train_docs[0], neg_ratio=4)
n_pos = sum(1 for _, _, l in sample_pairs if l == 1)
n_neg = sum(1 for _, _, l in sample_pairs if l == 0)
print(f"  Document: {train_docs[0]['id']}")
print(f"  Mentions: {len(train_docs[0]['mentions'])}")
print(f"  Positive pairs: {n_pos}")
print(f"  Negative pairs: {n_neg}")
print(f"  Ratio: 1:{n_neg/n_pos:.1f}" if n_pos > 0 else "  No positive pairs")
print("✓ Pair sampling verified")

Testing pair sampling...
  Document: 105_persuasion_brat_ben_Beng
  Mentions: 247
  Positive pairs: 3293
  Negative pairs: 13172
  Ratio: 1:4.0
✓ Pair sampling verified


In [14]:
"""
Cell 8: Threshold Tuning on Dev Set
"""

def tune_threshold(
    model: PairwiseCorefScorer,
    docs: List[Dict],
    threshold_range: np.ndarray
) -> Tuple[float, float, Dict]:
    """
    Tune clustering threshold on development set.
    
    Args:
        model: Trained pairwise scorer
        docs: Development documents
        threshold_range: Array of thresholds to try
    
    Returns:
        best_threshold, best_f1, all_results
    """
    model.pairwise_mlp.eval()
    
    # Pre-compute all pairwise scores for dev docs
    doc_scores = []
    for doc in tqdm(docs, desc="  Computing scores", leave=False):
        if len(doc['mentions']) < 2:
            doc_scores.append(None)
            continue
        
        embeddings = model.get_all_mention_embeddings(doc)
        scores = model.compute_pairwise_scores(embeddings)
        doc_scores.append(scores)
    
    # Try each threshold
    results = {}
    best_threshold = 0.5
    best_f1 = 0.0
    
    for threshold in threshold_range:
        all_gold = []
        all_pred = []
        offset = 0
        
        for doc, scores in zip(docs, doc_scores):
            if scores is None:
                continue
            
            pred_clusters = graph_connected_components(scores, threshold)
            gold_clusters = doc['clusters']
            
            # Offset for aggregate evaluation
            all_gold.extend([[m + offset for m in c] for c in gold_clusters])
            all_pred.extend([[m + offset for m in c] for c in pred_clusters])
            offset += len(doc['mentions'])
        
        metrics = CorefMetrics.evaluate(all_gold, all_pred)
        results[threshold] = metrics['CoNLL_F1']
        
        if metrics['CoNLL_F1'] > best_f1:
            best_f1 = metrics['CoNLL_F1']
            best_threshold = threshold
    
    return best_threshold, best_f1, results

print("✓ Threshold tuning function defined")
print(f"  Range: [{CONFIG.threshold_min}, {CONFIG.threshold_max}]")
print(f"  Step: {CONFIG.threshold_step}")

✓ Threshold tuning function defined
  Range: [0.1, 0.9]
  Step: 0.05


In [15]:
"""
Cell 9: Evaluation Function
"""

def evaluate_model(
    model: PairwiseCorefScorer,
    docs: List[Dict],
    threshold: float
) -> Dict[str, float]:
    """
    Evaluate model on a set of documents.
    
    Returns aggregate CoNLL metrics.
    """
    model.pairwise_mlp.eval()
    
    all_gold = []
    all_pred = []
    offset = 0
    
    for doc in tqdm(docs, desc="  Evaluating", leave=False):
        if len(doc['mentions']) < 2:
            continue
        
        # Get embeddings and scores
        embeddings = model.get_all_mention_embeddings(doc)
        scores = model.compute_pairwise_scores(embeddings)
        
        # Cluster
        pred_clusters = graph_connected_components(scores, threshold)
        gold_clusters = doc['clusters']
        
        # Aggregate
        all_gold.extend([[m + offset for m in c] for c in gold_clusters])
        all_pred.extend([[m + offset for m in c] for c in pred_clusters])
        offset += len(doc['mentions'])
    
    return CorefMetrics.evaluate(all_gold, all_pred)

print("✓ Evaluation function defined")

✓ Evaluation function defined


In [16]:
"""
Cell 10: Training Loop
"""

def train_model(
    model_config: ModelConfig,
    train_docs: List[Dict],
    dev_docs: List[Dict],
    config: TrainingConfig,
    device: str = 'cuda'
) -> Dict:
    """
    Train pairwise classifier for one model.
    
    Returns training results including:
    - best_threshold
    - dev_f1
    - training_history
    """
    print(f"\n{'='*70}")
    print(f"Training: {model_config.name}")
    print(f"{'='*70}")
    print(f"  HuggingFace ID: {model_config.hf_model_id}")
    print(f"  Category: {model_config.category}")
    
    start_time = time.time()
    
    # Load encoder
    print(f"\n  Loading encoder...")
    tokenizer = AutoTokenizer.from_pretrained(model_config.hf_model_id, trust_remote_code=True)
    encoder = AutoModel.from_pretrained(model_config.hf_model_id, trust_remote_code=True)
    encoder = encoder.to(device)
    
    # Create model
    model = PairwiseCorefScorer(
        encoder=encoder,
        tokenizer=tokenizer,
        hidden_dim=model_config.hidden_dim,
        mlp_hidden=config.mlp_hidden,
        dropout=config.mlp_dropout,
        device=device
    )
    
    # Optimizer (only MLP parameters)
    optimizer = torch.optim.AdamW(
        model.trainable_parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    # Loss
    criterion = nn.BCELoss()
    
    # Threshold range for tuning
    threshold_range = np.arange(
        config.threshold_min,
        config.threshold_max + config.threshold_step,
        config.threshold_step
    )
    
    # Training history
    history = {
        'epoch': [],
        'train_loss': [],
        'dev_f1': [],
        'best_threshold': []
    }
    
    best_dev_f1 = 0.0
    best_threshold = 0.5
    best_mlp_state = None
    patience_counter = 0
    
    print(f"\n  Training...")
    print(f"  {'Epoch':<8} {'Train Loss':<12} {'Dev F1':<10} {'Threshold':<10} {'Status'}")
    print(f"  {'-'*60}")
    
    for epoch in range(config.epochs):
        # =====================
        # TRAINING PHASE
        # =====================
        model.pairwise_mlp.train()
        
        # Create pairs for this epoch (re-sample negatives)
        epoch_pairs = create_epoch_pairs(train_docs, neg_ratio=config.neg_ratio)
        
        # Pre-compute embeddings for efficiency
        doc_embeddings = {}
        for doc in train_docs:
            if len(doc['mentions']) >= 2:
                doc_embeddings[doc['id']] = model.get_all_mention_embeddings(doc)
        
        # Training batches
        total_loss = 0.0
        n_batches = 0
        
        for batch_start in range(0, len(epoch_pairs), config.batch_size):
            batch = epoch_pairs[batch_start:batch_start + config.batch_size]
            
            # Prepare batch
            emb_i_list = []
            emb_j_list = []
            labels = []
            
            for doc, i, j, label in batch:
                if doc['id'] not in doc_embeddings:
                    continue
                emb = doc_embeddings[doc['id']]
                emb_i_list.append(emb[i])
                emb_j_list.append(emb[j])
                labels.append(label)
            
            if not labels:
                continue
            
            # Stack tensors
            emb_i = torch.stack(emb_i_list)
            emb_j = torch.stack(emb_j_list)
            labels_t = torch.tensor(labels, dtype=torch.float32, device=device).unsqueeze(1)
            
            # Forward pass
            optimizer.zero_grad()
            scores = model(emb_i, emb_j)
            loss = criterion(scores, labels_t)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / max(n_batches, 1)
        
        # =====================
        # EVALUATION PHASE
        # =====================
        model.pairwise_mlp.eval()
        
        # Tune threshold on dev
        threshold, dev_f1, _ = tune_threshold(model, dev_docs, threshold_range)
        
        # Record history
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(avg_loss)
        history['dev_f1'].append(dev_f1)
        history['best_threshold'].append(threshold)
        
        # Check for improvement
        status = ""
        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_threshold = threshold
            best_mlp_state = copy.deepcopy(model.pairwise_mlp.state_dict())
            patience_counter = 0
            status = "★ Best"
        else:
            patience_counter += 1
            status = f"patience={patience_counter}/{config.patience}"
        
        print(f"  {epoch+1:<8} {avg_loss:<12.4f} {dev_f1:<10.2f} {threshold:<10.2f} {status}")
        
        # Early stopping
        if patience_counter >= config.patience:
            print(f"  Early stopping at epoch {epoch + 1}")
            break
    
    # Load best model
    if best_mlp_state is not None:
        model.pairwise_mlp.load_state_dict(best_mlp_state)
    
    total_time = time.time() - start_time
    
    print(f"\n  Training complete in {total_time/60:.1f} minutes")
    print(f"  Best dev F1: {best_dev_f1:.2f}% at threshold {best_threshold:.2f}")
    
    return {
        'model': model,
        'best_threshold': best_threshold,
        'best_dev_f1': best_dev_f1,
        'history': history,
        'training_time': total_time
    }

print("✓ Training loop defined")

✓ Training loop defined


In [11]:
"""
Cell 11: Run Fine-Tuning for All Models
"""

print("=" * 80)
print("STARTING FINE-TUNING EXPERIMENT")
print("=" * 80)
print(f"\nTrain: {len(train_docs)} docs")
print(f"Dev: {len(dev_docs)} docs")
print(f"Test: {len(test_docs)} docs")
print(f"Models: {len(MODELS)}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

all_results = []
experiment_start = time.time()

for i, model_config in enumerate(MODELS):
    print(f"\n[{i+1}/{len(MODELS)}] {model_config.name}")
    
    # Train
    train_result = train_model(
        model_config=model_config,
        train_docs=train_docs,
        dev_docs=dev_docs,
        config=CONFIG,
        device=device
    )
    
    model = train_result['model']
    best_threshold = train_result['best_threshold']
    
    # Evaluate on test set
    print(f"\n  Evaluating on test set...")
    test_metrics = evaluate_model(model, test_docs, best_threshold)
    
    print(f"  Test CoNLL F1: {test_metrics['CoNLL_F1']:.2f}%")
    print(f"  MUC: {test_metrics['MUC_F1']:.2f}% | B³: {test_metrics['B3_F1']:.2f}% | CEAF: {test_metrics['CEAF_F1']:.2f}%")
    
    # Save checkpoint
    checkpoint_path = CHECKPOINT_DIR / f"{model_config.name.replace(' ', '_').replace('-', '_')}_mlp.pt"
    model.save_mlp(checkpoint_path)
    print(f"  Saved: {checkpoint_path}")
    
    # Store results
    all_results.append({
        'model_name': model_config.name,
        'category': model_config.category,
        'finetuned_f1': test_metrics['CoNLL_F1'],
        'best_threshold': best_threshold,
        'dev_f1': train_result['best_dev_f1'],
        'test_metrics': test_metrics,
        'training_time': train_result['training_time'],
        'history': train_result['history']
    })
    
    # Cleanup
    del model, train_result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

experiment_time = time.time() - experiment_start

print(f"\n{'='*80}")
print("EXPERIMENT COMPLETE")
print(f"Total time: {experiment_time/60:.1f} minutes")
print(f"{'='*80}")

STARTING FINE-TUNING EXPERIMENT

Train: 141 docs
Dev: 10 docs
Test: 71 docs
Models: 5
Device: cuda

[1/5] mBERT

Training: mBERT
  HuggingFace ID: google-bert/bert-base-multilingual-cased
  Category: multilingual

  Loading encoder...

  Training...
  Epoch    Train Loss   Dev F1     Threshold  Status
  ------------------------------------------------------------


  1        0.3196       64.73      0.60       ★ Best


  2        0.2763       64.25      0.85       patience=1/3


  3        0.2578       65.02      0.65       ★ Best


  4        0.2463       64.70      0.65       patience=1/3


...

  2        0.3655       65.51      0.50       ★ Best


  3        0.3579       64.50      0.25       patience=1/3


  4        0.3522       64.51      0.30       patience=2/3


  5        0.3494       63.49      0.40       patience=3/3
  Early stopping at epoch 5

  Training complete in 80.6 minutes
  Best dev F1: 65.51% at threshold 0.50

  Evaluating on test set...


  Test CoNLL F1: 63.06%
  MUC: 95.70% | B³: 66.22% | CEAF: 27.26%
  Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/checkpoints/BERT_Base_Uncased_mlp.pt

EXPERIMENT COMPLETE
Total time: 580.7 minutes


In [12]:
"""
Cell 12: Results Summary
"""

from IPython.display import display

print("\n" + "=" * 80)
print("FINE-TUNING RESULTS: FROZEN ENCODER + PAIRWISE CLASSIFIER")
print("=" * 80)
print(f"Training data: {TRAIN_FILE.name}")
print(f"Test set: {len(test_docs)} documents")

# Create summary dataframe
summary_data = []
for r in all_results:
    summary_data.append({
        'Model': r['model_name'],
        'Category': r['category'],
        'CoNLL F1': r['finetuned_f1'],
        'MUC F1': r['test_metrics']['MUC_F1'],
        'B³ F1': r['test_metrics']['B3_F1'],
        'CEAF F1': r['test_metrics']['CEAF_F1'],
        'Threshold': r['best_threshold'],
        'Dev F1': r['dev_f1'],
        'Train Time (min)': r['training_time'] / 60,
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('CoNLL F1', ascending=False).reset_index(drop=True)
df_summary.index = df_summary.index + 1
df_summary.index.name = 'Rank'

# Format for display
df_display = df_summary.copy()
df_display['CoNLL F1'] = df_display['CoNLL F1'].apply(lambda x: f"{x:.2f}%")
df_display['MUC F1'] = df_display['MUC F1'].apply(lambda x: f"{x:.2f}%")
df_display['B³ F1'] = df_display['B³ F1'].apply(lambda x: f"{x:.2f}%")
df_display['CEAF F1'] = df_display['CEAF F1'].apply(lambda x: f"{x:.2f}%")
df_display['Threshold'] = df_display['Threshold'].apply(lambda x: f"{x:.2f}")
df_display['Dev F1'] = df_display['Dev F1'].apply(lambda x: f"{x:.2f}%")
df_display['Train Time (min)'] = df_display['Train Time (min)'].apply(lambda x: f"{x:.1f}")

print("\n### Test Set Performance\n")
display(df_display[['Model', 'Category', 'CoNLL F1', 'MUC F1', 'B³ F1', 'CEAF F1']])

print("\n### Training Details\n")
display(df_display[['Model', 'Threshold', 'Dev F1', 'Train Time (min)']])

# Summary statistics
best_model = df_summary.iloc[0]['Model']
best_score = df_summary.iloc[0]['CoNLL F1']
avg_score = df_summary['CoNLL F1'].mean()

print(f"\n" + "-" * 60)
print(f"Best model: {best_model} ({best_score:.2f}%)")
print(f"Average CoNLL F1: {avg_score:.2f}%")
print(f"Total training time: {experiment_time/60:.1f} minutes")
print("-" * 60)


FINE-TUNING RESULTS: FROZEN ENCODER + PAIRWISE CLASSIFIER
Training data: transmucores_bencoref_train.conll
Test set: 71 documents

### Test Set Performance



,Model,Category,CoNLL F1,MUC F1,B³ F1,CEAF F1
Rank,,,,,,
1,MuRIL-Large,indic-focused,66.70%,96.56%,66.79%,36.74%
2,BanglaBERT-Base,bengali-specific,63.08%,96.34%,65.67%,27.23%
3,BERT-Base-Uncased,control,63.06%,95.70%,66.22%,27.26%
4,RemBERT,low-resource,62.80%,96.20%,65.67%,26.51%
5,mBERT,multilingual,62.33%,96.69%,67.11%,23.19%



### Training Details



,Model,Threshold,Dev F1,Train Time (min)
Rank,,,,
1,MuRIL-Large,0.55,68.77%,111.0
2,BanglaBERT-Base,0.60,67.33%,50.2
3,BERT-Base-Uncased,0.50,65.51%,80.6
4,RemBERT,0.55,66.14%,238.0
5,mBERT,0.85,65.21%,80.3



------------------------------------------------------------
Best model: MuRIL-Large (66.70%)
Average CoNLL F1: 63.59%
Total training time: 580.7 minutes
------------------------------------------------------------


In [21]:
"""
Add Per-Document Cluster Statistics to Results
FULLY SELF-CONTAINED VERSION - includes all metric functions
"""

import torch
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from scipy.optimize import linear_sum_assignment

print("="*80)
print("ADDING PER-DOCUMENT CLUSTER STATISTICS")
print("="*80)

# ============ METRIC FUNCTIONS (from Cell 4) ============

def muc(predicted_clusters, gold_clusters):
    """MUC metric (Vilain et al., 1995)."""
    def links(cluster):
        return len(cluster) - 1 if len(cluster) > 0 else 0
    
    def partition_numerator(key_clusters, response_clusters):
        count = 0
        for key_cluster in key_clusters:
            if len(key_cluster) == 0:
                continue
            partitions = 0
            for mention in key_cluster:
                for response_cluster in response_clusters:
                    if mention in response_cluster:
                        partitions += 1
                        break
            count += links(key_cluster) - (partitions - 1)
        return count
    
    pred_non_singleton = [c for c in predicted_clusters if len(c) > 1]
    gold_non_singleton = [c for c in gold_clusters if len(c) > 1]
    
    if not gold_non_singleton:
        return 100.0 if not pred_non_singleton else 0.0
    
    recall_num = partition_numerator(gold_non_singleton, pred_non_singleton)
    recall_den = sum(links(c) for c in gold_non_singleton)
    
    precision_num = partition_numerator(pred_non_singleton, gold_non_singleton)
    precision_den = sum(links(c) for c in pred_non_singleton)
    
    recall = recall_num / recall_den if recall_den > 0 else 0
    precision = precision_num / precision_den if precision_den > 0 else 0
    
    if precision + recall == 0:
        return 0.0
    return 200 * precision * recall / (precision + recall)

def b_cubed(predicted_clusters, gold_clusters):
    """B-cubed metric (Bagga and Baldwin, 1998)."""
    mention_to_gold = {}
    mention_to_pred = {}
    
    for cluster in gold_clusters:
        for mention in cluster:
            mention_to_gold[mention] = cluster
    
    for cluster in predicted_clusters:
        for mention in cluster:
            mention_to_pred[mention] = cluster
    
    all_mentions = set(mention_to_gold.keys()) | set(mention_to_pred.keys())
    
    if not all_mentions:
        return 0.0
    
    precision_sum = 0
    recall_sum = 0
    
    for mention in all_mentions:
        gold_cluster = mention_to_gold.get(mention, {mention})
        pred_cluster = mention_to_pred.get(mention, {mention})
        
        overlap = len(gold_cluster & pred_cluster)
        
        precision_sum += overlap / len(pred_cluster) if pred_cluster else 0
        recall_sum += overlap / len(gold_cluster) if gold_cluster else 0
    
    precision = precision_sum / len(all_mentions)
    recall = recall_sum / len(all_mentions)
    
    if precision + recall == 0:
        return 0.0
    return 200 * precision * recall / (precision + recall)

def ceaf_e(predicted_clusters, gold_clusters):
    """CEAF-e metric (Luo, 2005) - entity-based."""
    def phi(c1, c2):
        return len(c1 & c2)
    
    pred_list = [c for c in predicted_clusters if len(c) > 0]
    gold_list = [c for c in gold_clusters if len(c) > 0]
    
    if not gold_list:
        return 100.0 if not pred_list else 0.0
    if not pred_list:
        return 0.0
    
    # Cost matrix (negative for Hungarian algorithm which minimizes)
    cost_matrix = np.zeros((len(gold_list), len(pred_list)))
    for i, gold_c in enumerate(gold_list):
        for j, pred_c in enumerate(pred_list):
            cost_matrix[i, j] = -phi(gold_c, pred_c)
    
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    total_phi = -cost_matrix[row_ind, col_ind].sum()
    
    recall = total_phi / sum(len(c) for c in gold_list)
    precision = total_phi / sum(len(c) for c in pred_list)
    
    if precision + recall == 0:
        return 0.0
    return 200 * precision * recall / (precision + recall)

# ============ MAIN CODE ============

# Verify which experiment we're working on
print(f"\nExperiment directory: {OUTPUT_DIR}")

# Load existing results from JSON
results_file = OUTPUT_DIR / 'fine_tuning_results.json'
print(f"Loading results from: {results_file}")

if not results_file.exists():
    raise FileNotFoundError(f"Results file not found: {results_file}")

with open(results_file, 'r') as f:
    results_json = json.load(f)

all_results = results_json['results']
print(f"Found {len(all_results)} model results")

# Check checkpoints
checkpoint_dir = OUTPUT_DIR / 'checkpoints'
print(f"\nCheckpoint directory: {checkpoint_dir}")
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob('*.pt'))
    print(f"Found checkpoints: {[c.name for c in checkpoints]}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

def get_per_document_stats(model, docs, threshold):
    """Evaluate model and return per-document statistics."""
    model.pairwise_mlp.eval()
    per_document = []
    
    for doc in tqdm(docs, desc="  Collecting per-doc stats", leave=False):
        doc_id = doc['id']
        n_mentions = len(doc['mentions'])
        gold_clusters = doc['clusters']
        n_gold = len([c for c in gold_clusters if len(c) > 1])
        
        if n_mentions < 2:
            per_document.append({
                'doc_id': doc_id,
                'n_mentions': n_mentions,
                'n_gold_clusters': n_gold,
                'n_pred_clusters': 0,
                'CoNLL_F1': 0.0,
            })
            continue
        
        # Get embeddings and scores
        embeddings = model.get_all_mention_embeddings(doc)
        scores = model.compute_pairwise_scores(embeddings)
        
        # Cluster
        pred_clusters = graph_connected_components(scores, threshold)
        n_pred = len([c for c in pred_clusters if len(c) > 1])
        
        # Calculate metrics
        gold_for_metrics = [set(cluster) for cluster in gold_clusters]
        pred_for_metrics = [set(cluster) for cluster in pred_clusters]
        
        muc_f1 = muc(pred_for_metrics, gold_for_metrics)
        b3_f1 = b_cubed(pred_for_metrics, gold_for_metrics)
        ceaf_f1 = ceaf_e(pred_for_metrics, gold_for_metrics)
        doc_conll_f1 = (muc_f1 + b3_f1 + ceaf_f1) / 3
        
        per_document.append({
            'doc_id': doc_id,
            'n_mentions': n_mentions,
            'n_gold_clusters': n_gold,
            'n_pred_clusters': n_pred,
            'CoNLL_F1': doc_conll_f1,
        })
    
    return per_document

# Process each model
print(f"\nProcessing {len(all_results)} models on {len(test_docs)} test documents...\n")

for result in all_results:
    model_name = result['model_name']
    print(f"  {model_name}...")
    
    safe_name = model_name.replace('-', '_').replace(' ', '_')
    checkpoint_path = OUTPUT_DIR / 'checkpoints' / f"{safe_name}_mlp.pt"
    
    if not checkpoint_path.exists():
        print(f"    ✗ Checkpoint not found: {checkpoint_path}")
        continue
    
    # Find model config
    model_config = None
    for mc in MODELS:
        if mc.name == model_name:
            model_config = mc
            break
    
    if model_config is None:
        print(f"    ✗ Config not found for {model_name}")
        continue
    
    # Load encoder and model
    tokenizer = AutoTokenizer.from_pretrained(model_config.hf_model_id, trust_remote_code=True)
    encoder = AutoModel.from_pretrained(model_config.hf_model_id, trust_remote_code=True)
    encoder = encoder.to(device)
    encoder.eval()
    
    # Create model and load checkpoint
    model = PairwiseCorefScorer(encoder, tokenizer, model_config.hidden_dim, device=device)
    model.load_mlp(checkpoint_path)
    
    # Get per-document statistics
    threshold = result['best_threshold']
    per_doc_stats = get_per_document_stats(model, test_docs, threshold)
    
    # Add to results
    result['per_document'] = per_doc_stats
    
    # Summary
    avg_gold = np.mean([d['n_gold_clusters'] for d in per_doc_stats])
    avg_pred = np.mean([d['n_pred_clusters'] for d in per_doc_stats])
    print(f"    ✓ Avg gold: {avg_gold:.2f}, Avg pred: {avg_pred:.2f}")
    
    # Clean up
    del model, encoder
    torch.cuda.empty_cache()

# Save updated results
results_json['results'] = all_results

with open(results_file, 'w') as f:
    json.dump(results_json, f, indent=2)

print("\n" + "="*80)
print(f"✓ Updated results saved to: {results_file}")
print("="*80)

ADDING PER-DOCUMENT CLUSTER STATISTICS

Experiment directory: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned
Loading results from: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/fine_tuning_results.json
Found 5 model results

Checkpoint directory: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/checkpoints
Found checkpoints: ['BERT_Base_Uncased_mlp.pt', 'BanglaBERT_Base_mlp.pt', 'MuRIL_Large_mlp.pt', 'RemBERT_mlp.pt', 'mBERT_mlp.pt']
Device: cuda

Processing 5 models on 71 test documents...

  mBERT...


    ✓ Avg gold: 3.96, Avg pred: 1.82
  BanglaBERT-Base...


    ✓ Avg gold: 3.96, Avg pred: 1.46
  RemBERT...


    ✓ Avg gold: 3.96, Avg pred: 1.69
  MuRIL-Large...


Some weights of the model checkpoint at google/muril-large-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


    ✓ Avg gold: 3.96, Avg pred: 1.77
  BERT-Base-Uncased...


    ✓ Avg gold: 3.96, Avg pred: 2.04

✓ Updated results saved to: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/fine_tuning_results.json


In [13]:
"""
Cell 13: Save Results
"""

# Save summary CSV
summary_file = OUTPUT_DIR / 'fine_tuning_summary.csv'
df_summary.to_csv(summary_file)
print(f"Saved: {summary_file}")

# Save complete results JSON
results_json = {
    'experiment_name': 'Fine-tuning with Frozen Encoder + Pairwise Classifier',
    'date': datetime.now().isoformat(),
    'data': {
        'train_file': str(TRAIN_FILE),
        'train_docs': len(train_docs),
        'dev_file': str(DEV_FILE),
        'dev_docs': len(dev_docs),
        'test_file': str(TEST_FILE),
        'test_docs': len(test_docs),
    },
    'config': {
        'neg_ratio': CONFIG.neg_ratio,
        'learning_rate': CONFIG.learning_rate,
        'epochs': CONFIG.epochs,
        'batch_size': CONFIG.batch_size,
        'patience': CONFIG.patience,
        'threshold_range': f'[{CONFIG.threshold_min}, {CONFIG.threshold_max}], step={CONFIG.threshold_step}',
        'mlp_hidden': CONFIG.mlp_hidden,
        'mlp_dropout': CONFIG.mlp_dropout,
    },
    'methodology': {
        'encoder': 'Frozen (all layers)',
        'trainable': 'Pairwise MLP classifier',
        'loss': 'Binary Cross-Entropy',
        'negative_sampling': f'1:{CONFIG.neg_ratio} ratio, re-sampled each epoch',
        'threshold_tuning': 'Dev set only (no test tuning)',
        'clustering': 'Graph-based Connected Components',
    },
    'results': [
        {
            'model_name': r['model_name'],
            'category': r['category'],
            'test_conll_f1': r['finetuned_f1'],
            'test_muc_f1': r['test_metrics']['MUC_F1'],
            'test_b3_f1': r['test_metrics']['B3_F1'],
            'test_ceaf_f1': r['test_metrics']['CEAF_F1'],
            'best_threshold': r['best_threshold'],
            'dev_f1': r['dev_f1'],
            'training_time_seconds': r['training_time'],
        }
        for r in all_results
    ],
    'total_time_minutes': experiment_time / 60,
}

results_file = OUTPUT_DIR / 'fine_tuning_results.json'
with open(results_file, 'w') as f:
    json.dump(results_json, f, indent=2, default=float)
print(f"Saved: {results_file}")

# Save training histories
for r in all_results:
    history_df = pd.DataFrame(r['history'])
    history_file = OUTPUT_DIR / f"history_{r['model_name'].replace(' ', '_').replace('-', '_')}.csv"
    history_df.to_csv(history_file, index=False)
    print(f"Saved: {history_file}")

print(f"\n✓ All results saved to {OUTPUT_DIR}")

Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/fine_tuning_summary.csv
Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/fine_tuning_results.json
Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/history_mBERT.csv
Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/history_BanglaBERT_Base.csv
Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/history_RemBERT.csv
Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/history_MuRIL_Large.csv
Saved: /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/history_BERT_Base_Uncased.csv

✓ All results saved to /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned


In [14]:
"""
Cell 14: Conclusion
"""

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)

print(f"""
EXPERIMENT DETAILS
──────────────────
Training data: {TRAIN_FILE.name}
Training documents: {len(train_docs)}
Dev documents: {len(dev_docs)}
Test documents: {len(test_docs)}

METHODOLOGY
───────────
- Encoder: Frozen (preserves pre-trained representations)
- Trainable: Pairwise MLP classifier
- Loss: Binary Cross-Entropy
- Negative sampling: 1:{CONFIG.neg_ratio} ratio, re-sampled each epoch
- Threshold: Tuned on dev set [{CONFIG.threshold_min}, {CONFIG.threshold_max}]
- Clustering: Graph-based Connected Components

RESULTS
───────
- Best model: {best_model}
- Best CoNLL F1: {best_score:.2f}%
- Average CoNLL F1: {avg_score:.2f}%

OUTPUT FILES
────────────
- {OUTPUT_DIR / 'fine_tuning_summary.csv'}
- {OUTPUT_DIR / 'fine_tuning_results.json'}
- {CHECKPOINT_DIR}/*.pt (model checkpoints)
- {OUTPUT_DIR}/history_*.csv (training histories)
""")
print("=" * 80)


EXPERIMENT COMPLETE

EXPERIMENT DETAILS
──────────────────
Training data: transmucores_bencoref_train.conll
Training documents: 141
Dev documents: 10
Test documents: 71

METHODOLOGY
───────────
- Encoder: Frozen (preserves pre-trained representations)
- Trainable: Pairwise MLP classifier
- Loss: Binary Cross-Entropy
- Negative sampling: 1:4 ratio, re-sampled each epoch
- Threshold: Tuned on dev set [0.1, 0.9]
- Clustering: Graph-based Connected Components

RESULTS
───────
- Best model: MuRIL-Large
- Best CoNLL F1: 66.70%
- Average CoNLL F1: 63.59%

OUTPUT FILES
────────────
- /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/fine_tuning_summary.csv
- /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/fine_tuning_results.json
- /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/checkpoints/*.pt (model checkpoints)
- /teamspace/studios/this_studio/final_experiments/experiment_1_finetuned/history_*.csv (training histories)

